# What if a neural network learned the taste factors?

Unit 5 learned a recommender the hard way and the honest way: a reader factor `P_u` and a book
factor `Q_i`, found by full-batch **numpy gradient descent** on a pointwise logistic loss, scoring a
match as the dot product `P_u . Q_i`. Unit 7 did not learn its vectors at all -- it *borrowed* GloVe
word vectors pretrained on a mountain of text. Both retrieve the same way: a **dot product between
embeddings**.

This unit asks the Part-2 question head on: what if we let a **neural network** learn those
embeddings directly from the interaction log? That is the **two-tower** (or dual-encoder) model -- a
reader-ID embedding tower and a book-ID embedding tower, scored by a dot product, trained in
**PyTorch**. It is Unit 5's matrix factorization *re-expressed as a learned neural retriever*, and it
turns out to be the **best path in the book so far** -- but only when we regularize it, and the
reason why is the real lesson of the unit.

We will build it from scratch: the architecture, the pairwise ranking loss and one training step by
hand, then the full towers in PyTorch, then the shipped `TwoTowerRetrievalPath`. Along the way we
meet the knob -- **weight decay** -- that decides whether the model tops matrix factorization or
collapses below collaborative filtering.

## Load the substrate and fit the comparison paths

The Units 1-7 substrate is unchanged: `generated_dir()` locates the seeded slice, the catalog and the
gzip'd interaction log load as before, and `cold_partitions.json` lists the cold readers we always
exclude from scoring. We fit the five comparison paths now -- the random floor, popularity, lexical
BM25, item-item collaborative filtering, and Unit 5's matrix factorization -- so the two-tower has an
honest field to be measured against later. (`test` stays sealed for Checkpoint A; we only ever score
on `val`.)

In [ ]:
import json

import numpy as np
import pandas as pd
from bookrec import (
    ItemItemRetrievalPath,
    LexicalRetrievalPath,
    MatrixFactorizationPath,
    PopularityRetrievalPath,
    RandomRetrievalPath,
    TwoTowerRetrievalPath,
    generated_dir,
    load_catalog,
    load_keywords,
    run_validation_scoreboard,
)

data_dir = generated_dir()
catalog = load_catalog(data_dir / "catalog.csv.gz")
catalog_ids = sorted(catalog)
interactions_path = data_dir / "interactions.csv.gz"
rows = pd.read_csv(interactions_path).to_dict("records")
keywords = load_keywords(data_dir / "keywords.csv.gz")
cold_readers = set(json.loads((data_dir / "cold_partitions.json").read_text())["cold_readers"])

random_path = RandomRetrievalPath(catalog_ids, seed=0)
popularity = PopularityRetrievalPath().fit(rows, catalog=catalog_ids)
lexical = LexicalRetrievalPath(keywords).fit(rows, catalog=catalog_ids)
item_item = ItemItemRetrievalPath().fit(rows, catalog=catalog_ids)
mf = MatrixFactorizationPath(seed=0).fit(rows, catalog=catalog_ids)

print(f"catalog {len(catalog_ids)} books; {len(rows)} interaction rows; {len(cold_readers)} cold readers")
print("fitted: random, popularity, lexical, item-item CF, matrix factorization")

## 1. The two-tower is matrix factorization's dot product, re-expressed

Unit 5's matrix factorization stored one learned vector per reader (`P_u`) and one per book (`Q_i`),
and scored a reader-book match as their dot product. The **two-tower** architecture is exactly that,
drawn as two neural *towers*:

- a **reader tower** -- a lookup table with one learned vector per reader ID, and
- a **book tower** -- a lookup table with one learned vector per book ID.

To score a (reader, book) pair, each tower looks up its vector and we take the **dot product**:
`score(reader, book) = reader_emb . book_emb`. That lookup table is PyTorch's `nn.Embedding`: a
matrix of learned numbers where row `i` *is* the embedding of item `i`, and indexing it with an ID
returns that row. Nothing about the *model* is new -- it is the same latent dot product as Unit 5's
MF and Unit 7's GloVe cosine. What is new is *how we learn it*: autograd and an optimizer instead of
hand-written numpy gradients. Build two tiny towers and score one pair, to see the mechanism.

In [ ]:
import torch

# Determinism knobs (Unit's reproducibility lesson): a fixed seed makes the random init repeatable,
# and single-thread + deterministic algorithms make every run on this CPU produce identical numbers.
torch.manual_seed(0)
torch.use_deterministic_algorithms(True)
torch.set_num_threads(1)

# A toy world: 3 readers, 4 books, 2-dimensional embeddings. Each tower is a lookup table of vectors.
reader_tower = torch.nn.Embedding(3, 2)
book_tower = torch.nn.Embedding(4, 2)
torch.nn.init.normal_(reader_tower.weight, mean=0.0, std=0.1)
torch.nn.init.normal_(book_tower.weight, mean=0.0, std=0.1)

# Look up reader 0's vector and book 2's vector (indexing the table returns that row)...
reader_vec = reader_tower(torch.from_numpy(np.array([0], dtype=np.int64)))
book_vec = book_tower(torch.from_numpy(np.array([2], dtype=np.int64)))

# ...and the match score is their dot product -- the same operation MF and GloVe scored with.
score = (reader_vec * book_vec).sum()
print(f"reader 0 embedding: {reader_vec.detach().numpy().round(3)}")
print(f"book 2   embedding: {book_vec.detach().numpy().round(3)}")
print(f"score (dot product): {score.item():.4f}")
print("These vectors are random right now -- training is what makes the dot product meaningful.")

## 2. The training signal: a pairwise ranking loss (BPR)

How do we train the towers? The log gives us **implicit feedback** -- a read or a like is a positive,
and there are no negative *ratings*, only books a reader has not touched. So, exactly as Unit 5 did,
we build **triples**: for each observed (reader, positive book), we sample a **negative** book the
reader has not read. We then ask the model for one thing: score the positive **above** the negative.

That is **Bayesian Personalized Ranking (BPR)**. With `s_pos = reader_emb . pos_emb` and
`s_neg = reader_emb . neg_emb`, the loss on one triple is

```
loss = -log sigmoid(s_pos - s_neg)
```

When the positive already outscores the negative the margin `s_pos - s_neg` is large and positive,
`sigmoid` is near 1, and the loss is near 0. When the negative wins, the margin is negative and the
loss is large. Unlike Unit 5's *pointwise* logistic loss (which pushed each score toward a 0/1
label), BPR is a *pairwise ranking* loss -- it only cares that the right book ranks higher, which is
exactly what top-k retrieval is judged on. Derive one gradient step by hand on three tiny vectors.

In [ ]:
# Three 2-D vectors: one reader, the book she liked (positive), a book she has not read (negative).
u = np.array([0.6, 0.3], dtype=np.float32)   # reader
i = np.array([0.5, -0.1], dtype=np.float32)  # positive book
j = np.array([0.4, 0.5], dtype=np.float32)   # negative book


def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))


s_pos = u @ i
s_neg = u @ j
margin = s_pos - s_neg
loss = -np.log(sigmoid(margin))
print(f"s_pos {s_pos:+.4f}   s_neg {s_neg:+.4f}   margin {margin:+.4f}   loss {loss:.4f}")
print("The negative currently outscores the positive -- the model has it backwards.")

# BPR gradient by hand. With m = s_pos - s_neg, d loss / d m = -(1 - sigmoid(m)), and the chain rule
# through the dot products gives the gradient for each vector:
g = -(1.0 - sigmoid(margin))
grad_u = g * (i - j)
grad_i = g * u
grad_j = g * (-u)
print(f"grad_u {grad_u.round(4)}   grad_i {grad_i.round(4)}   grad_j {grad_j.round(4)}")

# One gradient-descent step (learning rate 0.2) and re-score: the margin should rise.
lr = 0.2
u_next = u - lr * grad_u
i_next = i - lr * grad_i
j_next = j - lr * grad_j
margin_next = (u_next @ i_next) - (u_next @ j_next)
print(f"margin before {margin:+.4f}  ->  after one step {margin_next:+.4f}")

In [ ]:
# PyTorch's autograd computes that same gradient for us -- no calculus by hand. We flag the three
# vectors as things to differentiate, build the loss, and call .backward(); torch fills in .grad.
u_t = torch.from_numpy(u)
u_t.requires_grad = True
i_t = torch.from_numpy(i)
i_t.requires_grad = True
j_t = torch.from_numpy(j)
j_t.requires_grad = True

bpr = -torch.nn.functional.logsigmoid((u_t * i_t).sum() - (u_t * j_t).sum())
bpr.backward()

print(f"torch loss {bpr.item():.4f}  (hand-computed loss was {loss:.4f})")
print(f"autograd grad_u {u_t.grad.numpy().round(4)}  (hand-derived {grad_u.round(4)})")
print(f"gradients match -- u: {np.allclose(u_t.grad.numpy(), grad_u)}, "
      f"i: {np.allclose(i_t.grad.numpy(), grad_i)}, j: {np.allclose(j_t.grad.numpy(), grad_j)}")
print("Autograd reproduces the hand calculus exactly -- that is all .backward() is doing.")

### Repeat the step and watch the positive overtake the negative

One step nudged the margin up. BPR training is just that step, over and over, across every triple.
Run eight steps on our three toy vectors and watch `s_pos` climb while `s_neg` falls -- the positive
book overtakes the negative, and the loss slides down.

In [ ]:
u = np.array([0.6, 0.3], dtype=np.float32)
i = np.array([0.5, -0.1], dtype=np.float32)
j = np.array([0.4, 0.5], dtype=np.float32)
lr = 0.2
for step in range(8):
    s_pos = u @ i
    s_neg = u @ j
    margin = s_pos - s_neg
    loss = -np.log(sigmoid(margin))
    print(f"step {step}   s_pos {s_pos:+.3f}   s_neg {s_neg:+.3f}   margin {margin:+.3f}   loss {loss:.3f}")
    g = -(1.0 - sigmoid(margin))
    grad_u, grad_i, grad_j = g * (i - j), g * u, g * (-u)   # all from the CURRENT vectors
    u = u - lr * grad_u
    i = i - lr * grad_i
    j = j - lr * grad_j
print("The margin flips positive after one step; by step 7 the positive clearly outranks the negative.")

## 3. Scale the step up to the full towers

The toy step had one triple and two dimensions. The real model is the *same step* run over every
(reader, positive, negative) triple in the training log, with 32-dimensional embeddings, batched for
speed, and handed to the **Adam** optimizer instead of a fixed learning rate. Adam adapts the step
size per parameter, so training converges in far fewer passes than Unit 5's plain gradient descent.

Build the towers the same way the shipped path does -- `nn.Embedding` towers, small seeded init, Adam
over both towers' parameters, BPR over mini-batches of triples -- and train a few epochs to watch the
mean loss fall. (This short demo samples one uniform negative per positive; the shipped path does the
leakage-safe, collision-resampled sampling for real.)

In [ ]:
# Training triples: every (reader, book) positive from the TRAIN split (leakage-safe -- val/test and
# non-positive rows never enter training).
positives = [
    (int(row["reader_id"]), int(row["item_id"]))
    for row in rows
    if row["split"] == "train" and int(row["label"]) == 1
]
reader_ids = sorted({reader for reader, _ in positives})
reader_of = {reader: index for index, reader in enumerate(reader_ids)}
book_of = {book: index for index, book in enumerate(catalog_ids)}
pos_r = np.array([reader_of[reader] for reader, _ in positives], dtype=np.int64)
pos_b = np.array([book_of[book] for _, book in positives], dtype=np.int64)
n_readers, n_books, n_pos = len(reader_ids), len(catalog_ids), len(positives)
print(f"{n_readers} readers, {n_books} books, {n_pos} train positives")

torch.manual_seed(0)
reader_tower = torch.nn.Embedding(n_readers, 32)
book_tower = torch.nn.Embedding(n_books, 32)
torch.nn.init.normal_(reader_tower.weight, mean=0.0, std=0.1)
torch.nn.init.normal_(book_tower.weight, mean=0.0, std=0.1)
optimizer = torch.optim.Adam(
    list(reader_tower.parameters()) + list(book_tower.parameters()),
    lr=0.01, weight_decay=1e-4,
)

rng = np.random.default_rng(0)
batch_size = 256
for epoch in range(6):
    negatives = rng.integers(0, n_books, size=n_pos)   # one uniform negative per positive (demo)
    order = rng.permutation(n_pos)
    readers_t = torch.from_numpy(pos_r[order])
    pos_t = torch.from_numpy(pos_b[order])
    neg_t = torch.from_numpy(negatives[order])
    epoch_loss, n_batches = 0.0, 0
    for start in range(0, n_pos, batch_size):
        reader_vec = reader_tower(readers_t[start:start + batch_size])
        s_pos = (reader_vec * book_tower(pos_t[start:start + batch_size])).sum(dim=1)
        s_neg = (reader_vec * book_tower(neg_t[start:start + batch_size])).sum(dim=1)
        loss = -torch.nn.functional.logsigmoid(s_pos - s_neg).mean()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()   # .item() reads the number; not float(loss) on a grad tensor
        n_batches += 1
    print(f"epoch {epoch}   mean BPR loss {epoch_loss / n_batches:.4f}")

### Reveal: `TwoTowerRetrievalPath`

That loop -- `nn.Embedding` towers, Adam, mini-batched BPR -- *is* the shipped path, run for 60 epochs
with 10 negatives per positive, leakage-safe collision-resampled negative sampling, and the
determinism knobs saved and restored around training so later cells are unaffected. Fitting it is one
line, and it behaves like every other `RetrievalPath`: `fit` then `retrieve`. The path keeps a
`loss_history` so we can watch training converge.

In [ ]:
two_tower = TwoTowerRetrievalPath(weight_decay=1e-4).fit(rows, catalog=catalog_ids)
history = two_tower.loss_history
print(f"trained {len(history)} epochs")
print(f"mean BPR loss: epoch 0 = {history[0]:.4f}  ->  final epoch = {history[-1]:.4f}")
print(f"reader embeddings {two_tower.reader_embeddings.shape}, "
      f"book embeddings {two_tower.item_embeddings.shape}")
print("The loss falls and settles -- it does NOT crash to 0 (we will see why that matters).")

### The recommendation is still a dot product of learned embeddings

Retrieval has not changed since Unit 5: score every catalog book by `reader_emb . book_emb`, drop the
books the reader has already seen, keep the top k. Confirm that the path's score for a reader's top
book is exactly the dot product of the two learned embeddings -- the two-tower is a dot-product
retriever, end to end.

In [ ]:
from collections import defaultdict

seen_by_reader = defaultdict(set)
for row in rows:
    if row["split"] == "train" and int(row["label"]) == 1:
        seen_by_reader[int(row["reader_id"])].add(int(row["item_id"]))

reader_id = sorted(seen_by_reader)[0]
history_books = seen_by_reader[reader_id]
recommendations = two_tower.retrieve(reader_id, {"seen": history_books}, 5)
print(f"reader {reader_id} has read {len(history_books)} books; top 5 two-tower recommendations:")
for candidate in recommendations:
    print(f"  book {candidate.item_id:>5}   score {candidate.score:.4f}")

# Recompute the top book's score by hand from the learned embeddings.
reader_row = {reader: index for index, reader in enumerate(sorted(seen_by_reader))}[reader_id]
reader_emb = two_tower.reader_embeddings[reader_row]
book_scores = reader_emb @ two_tower.item_embeddings.T
top_book = recommendations[0].item_id
top_col = {book: index for index, book in enumerate(catalog_ids)}[top_book]
print(f"by-hand reader_emb . book_emb for book {top_book}: {book_scores[top_col]:.4f} "
      f"== path score {recommendations[0].score:.4f}")

## 3b. Weight decay: the knob that decides win or collapse

Here is the pivotal beat. Our training log has only 5,303 positives but the towers hold
`(540 + 2000) x 32` ~= 81,000 numbers -- far more capacity than the data needs. With nothing holding
it back, BPR can simply **memorize** the exact training triples: drive every `s_pos - s_neg` sky-high,
the loss to ~0, and learn nothing that generalizes to the held-out `val` books.

**Weight decay** (Adam's L2 penalty, `weight_decay`) fights that by pulling every embedding number
gently toward 0 each step, so the model can only grow the directions that actually help. Train the
*same* towers with `weight_decay=0` and watch the loss collapse toward 0 -- the signature of
overfitting.

In [ ]:
two_tower_plain = TwoTowerRetrievalPath(weight_decay=0.0).fit(rows, catalog=catalog_ids)
plain_history = two_tower_plain.loss_history
print("weight_decay = 0 (no regularization):")
print(f"  mean BPR loss: epoch 0 = {plain_history[0]:.4f}  ->  final epoch = {plain_history[-1]:.4f}")
print("weight_decay = 1e-4 (the pinned default):")
print(f"  mean BPR loss: epoch 0 = {history[0]:.4f}  ->  final epoch = {history[-1]:.4f}")
print()
print("Without weight decay the loss crashes toward 0 -- the model has memorized the train triples.")
print("With weight decay it settles well above 0 -- it kept only what generalizes. The scoreboard tells")
print("us which one actually recommends better.")

## Score honestly on the validation split

Now the real question, answered the same way every unit answers it: run the frozen `val` scoreboard
over the whole field -- the random floor, popularity, lexical BM25, item-item CF, Unit 5's MF, and
both two-towers (regularized and not) -- at hit@10, reusing the paths we already fit so we train
nothing twice.

In [ ]:
board = {
    "random": random_path,
    "popularity": popularity,
    "lexical": lexical,
    "item-item": item_item,
    "mf": mf,
    "two-tower (wd=1e-4)": two_tower,
    "two-tower (wd=0)": two_tower_plain,
}
print(f"{'path':>20} {'hit@10':>8}")
for name, path in board.items():
    result = run_validation_scoreboard(
        path, interactions_path, catalog_ids=catalog_ids, k=10, cold_readers=cold_readers
    )
    print(f"{name:>20} {result.hit_rate_at_k:>8.4f}")

Read the numbers honestly -- and this time the headline is a **win**. The regularized two-tower lands
at hit@10 **~0.340**, clearing Unit 5's matrix factorization (**0.276**), item-item CF (**0.252**),
lexical (**0.158**), popularity (**0.108**) and the random floor (**0.012**). It is the **best path
in the book so far**.

Why does it win, when it is the *same* reader-book dot product as MF? Two things changed, and neither
is "neural is magic":

- **The objective.** MF minimized a *pointwise* logistic loss (push each score toward a 0/1 label);
  the two-tower minimizes a *pairwise* BPR loss (rank the positive above a negative). Top-k retrieval
  is a ranking problem, and the ranking objective optimizes exactly what the scoreboard measures.
- **The optimizer + regularization.** Adam's adaptive steps plus weight decay find a better-ranking,
  better-generalizing set of embeddings than full-batch gradient descent did.

And the unregularized tower proves the point from the other side: at `weight_decay=0` it scores
**~0.144** -- *below* CF and even below lexical. Same architecture, same optimizer, same 60 epochs;
the only difference is the memorization we watched in the loss. Regularization is not a footnote here,
it is the difference between the book's best path and one that is worse than a Unit 4 baseline. (As
always, this is a result on *this* data at *this* pinned config, read once on `val`, with `test` left
sealed.)

## Where this goes

- The **two-tower / dual-encoder** is a reader-ID embedding tower and a book-ID embedding tower scored
  by a **dot product** -- Unit 5's matrix factorization re-expressed as a learned neural model.
- **BPR** is a pairwise ranking loss, `-log sigmoid(s_pos - s_neg)`, trained on (reader, positive,
  sampled-negative) triples. We derived one step by hand, confirmed autograd reproduces it, and
  scaled it to the full towers with `nn.Embedding` + Adam + mini-batches.
- Measured on `val`, the regularized two-tower (hit@10 **~0.340**) is the **book's new best path**,
  above MF (0.276) and CF (0.252) -- because the ranking objective optimizes top-k directly. Drop
  `weight_decay` and it overfits to **~0.144**, below CF: the headline lesson of the unit.

### The bridge ahead

Three units, one retriever: Unit 5's MF *learned* embeddings with numpy gradient descent, Unit 7
*borrowed* GloVe's pretrained ones, and Unit 8 *learned* them with a neural two-tower. All three score
the same way -- a dot product between a reader embedding and a book embedding. The towers here only
know an ID, so a brand-new book with no interactions gets no useful vector. **Unit 9** fixes that by
feeding each tower the item's *features*, so it can embed a cold book it has never seen; **Units
11-12** build rerankers and sequence models on top of this same learned embedding stack. The
retriever stays a dot product; what we feed the towers keeps getting richer.